In [1]:
from torch import nn
import torch
from d2l import torch as d2l

In [3]:
def masked_softmax(X,valid_lens):
    if valid_lens is None:
        return nn.functional.softmax(X,dim=-1)
    if valid_lens.dim()==1:
        valid_lens=torch.repeat_interleave(valid_lens,X.shape[1])
    else:
        valid_lens=valid_lens.reshape(-1)
    shape = X.shape
    X = d2l.sequence_mask(X.reshape(-1, shape[-1]), valid_lens,  value=-1e6)

    return torch.softmax(X,dim=-1).reshape(shape)


class AddtiveAttention(nn.Module):
    def __init__(self,query_size,key_size,hidden_size,drop_out):
        super().__init__()
        self.W_k=nn.Linear(key_size,hidden_size,bias=False)
        self.W_q=nn.Linear(query_size,hidden_size,bias=False)
        self.W_v=nn.Linear(hidden_size,1,bias=False)
        self.drop_out=nn.Dropout(drop_out)

    def forward(self,query:torch.Tensor,key:torch.Tensor,value:torch.Tensor,valid_lens:torch.Tensor):
        '''
        query:[batch_size,num_q,query_size]
        key:[batch_size,num_k,key_size]
        value:[batch_size,num_k,value_size]
        '''
        query_h,key_h=self.W_q(query),self.W_k(key)
        query_h=torch.unsqueeze(query_h,dim=2)
        key_h=torch.unsqueeze(key_h,dim=1)
        hid=torch.tanh(query_h+key_h)
        attention_weight=masked_softmax(self.W_v(hid).squeeze(-1),valid_lens)

        return torch.bmm(self.drop_out(attention_weight),value)


query=torch.randn([1,5,5])
key=torch.randn([1,6,7])
value=torch.randn([1,6,9])
valid_lens=torch.tensor([5])
aa=AddtiveAttention(query_size=5,key_size=7,hidden_size=10,drop_out=0.1)
aa(query,key,value,valid_lens)

tensor([[[-0.3850, -1.1906,  0.0416, -0.1404, -0.1414, -0.4815,  0.1140,
           0.1436,  0.7046],
         [-0.3874, -1.1907,  0.0622, -0.1624, -0.1566, -0.4829,  0.1424,
           0.1476,  0.6822],
         [-0.3947, -1.1842,  0.0677, -0.2090, -0.2072, -0.4771,  0.1913,
           0.1663,  0.6428],
         [-0.3764, -1.1900,  0.0299, -0.1427, -0.1317, -0.4801,  0.1186,
           0.1456,  0.7094],
         [-0.3864, -1.1835,  0.1013, -0.2480, -0.2195, -0.4765,  0.2430,
           0.1641,  0.6255]]], grad_fn=<BmmBackward0>)

In [ ]:
class decoder(d2l.decoder):
    def __init__(self,vocab_size,embed_size,hidden_size,num_layers):
        self.vocab_size=vocab_size
        self.embed_size=embed_size
        self.hidden_size=hidden_size
        self.num_layers=num_layers

        self.addtive_attention=AddtiveAttention(hidden_size,hidden_size,hidden_size)
        self.embedding=nn.Embedding(vocab_size,embed_size,dtype=torch.float16)
        self.rnn=nn.GLU(self.embed_size+self.hidden_size,hidden_size,hidden_size,num_layers,dtype=torch.float16)
        self.Linear=nn.Linear(self.hidden_size,self.vocab_size,dtype=torch.float16)
        return
    def init_state(self,enc_outputs,hidden_state):
        ''' enc_outputs:[num_steps,batch_size,hidden_size]
            hidden_state:[num_layers,batch_size,hidden_size]
        '''
        return (torch.permute(enc_outputs,[1,0,2]),hidden_state)

    def forward(self,X,state):
        '''
        X=[batch_size,num_step,vocab_size]
        state=([batch_size,num_steps,hidden_size],[num_layers,batch_size,hidden_size])
        '''
        enc_outputs,hidden_state=state
        X=self.embedding(X).permute([1,0,2])
        outputs,self._attention_weight=([],[])
        for x in X:
            context=self.addtive_attention(hidden_state[-1].unsqueeze(1),enc_outputs,enc_outputs)
            x_in=torch.cat((x,context),dim=1)
            out,hidden_state=self.rnn(x_in,hidden_state)
            outputs.append(out)
            self._attention_weight.append(context)
        outputs=self.Linear(torch.cat(outputs,dim=0))
        return outputs.permute([1,0,2]),(enc_outputs,hidden_state)

    @property
    def attention_weight(self):
        return self._attention_weight

In [ ]:
encoder=d2l.Seq2SeqEncoder(vocab_size=10,embed_size=8,hidden_size=16,num_layers=2)
decoder=decoder(vocab_size=10,embed_size=8,hidden_size=16,num_layers=2)
encoder.eval()
decoder.eval()
X=torch.zeros((4,7),dtype=torch.long)
state = decoder.init_state(encoder(X), None) 
output, state = decoder(X, state) 
output.shape, len(state), state[0].shape, len(state[1]), state[1][0].shape